In [ ]:
from ollama import chat

response = chat(
    model='phi3.5',
    messages=[
        {
            'role': 'user',
            'content': 'Hello, introduce yourself'
        }
    ]
)

print(response['message']['content'])

In [ ]:
import json
import pdfplumber
from ollama import chat

# =====================================================
# FILE PATH
# =====================================================

FILE_PATH = r"D:\365\data\BẢNG-GIÁ-KHUYẾN-MÃI-GIAI-ĐOẠN-10.02-ĐẾN-31.03.2023 (1).pdf"

# =====================================================
# READ PDF
# =====================================================

def read_pdf(pdf_path):

    text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    return text

# =====================================================
# LOAD PDF TEXT
# =====================================================

pdf_text = read_pdf(FILE_PATH)

print("=" * 50)
print("PDF TEXT PREVIEW")
print("=" * 50)

print(pdf_text[:3000])

# =====================================================
# PROMPT
# =====================================================

PROMPT = f"""
You are an expert hospitality FIT contract extraction engine.

Your task is to extract ONLY FIT (Free Independent Traveler)
hotel contract information from the uploaded document.

Return STRICT VALID JSON ONLY.

==================================================
CORE EXTRACTION RULES
==================================================

1. ONLY extract information explicitly written in the document.

2. DO NOT hallucinate, infer, estimate, normalize, or invent values.

3. IGNORE all NON-FIT information:
- GIT
- GROUP
- SERIES
- CORPORATE
- WHOLESALE
- MICE
- EVENT
- ALLOTMENT
- CREW
- CONTRACTOR
- OTA CAMPAIGN

4. If a table contains both FIT and NON-FIT data:
extract ONLY FIT rows and columns.

5. USE ONLY the EXACT schema keys defined below.

6. DO NOT:
- create additional keys
- rename keys
- restructure arrays
- restructure objects
- generate nested attributes not defined in schema

7. NEVER generate undefined keys such as:
- age_bracket
- breakfast_fee
- room_rate_fit
- policy_item
- surcharge_data
- hotel_package
- package_detail
- extra_information

8. Preserve EXACTLY:
- room names
- currencies
- VND formatting
- percentages
- date ranges
- rate formatting
- punctuation
- multilingual wording

9. If information is missing:
- use empty string ""
- use empty array []
- NEVER guess

10. NEVER explain.

11. NEVER summarize.

12. NEVER return markdown.

13. NEVER wrap JSON inside code blocks.

14. Schema consistency is CRITICAL across all hotels.

15. Return STRICT VALID JSON ONLY.

==================================================
SEMANTIC RULES
==================================================

A. facilities
= permanent hotel facilities/amenities.

Examples:
- swimming pool
- gym
- spa
- sauna
- wifi
- parking
- beach access
- kids club

B. included_benefits
= benefits included with booking/rate/package.

Examples:
- breakfast included
- airport transfer
- welcome drink
- complimentary dinner
- spa voucher
- late checkout

DO NOT duplicate facilities into included_benefits.

C. meal_information
ONLY use for:
- restaurant information
- gala dinner
- compulsory dinner
- buffet pricing
- special meal pricing

DO NOT duplicate breakfast inclusion here
if breakfast already exists inside room seasonal_rates.

Breakfast inclusion inside room seasonal_rates
is the PRIMARY SOURCE OF TRUTH.

==================================================
OUTPUT STRUCTURE
==================================================

{{
  "hotel_information": {{
    "hotel_name": "",
    "brand_group": "",
    "address": "",
    "district": "",
    "city": "",
    "country": "",
    "website": "",
    "email": [],
    "phone_numbers": [],
    "tax_code": "",
    "representatives": [
      {{
        "name": "",
        "position": ""
      }}
    ]
  }},

  "contract_information": {{
    "contract_name": "",
    "contract_number": "",
    "validity": {{
      "start_date": "",
      "end_date": ""
    }},
    "applicable_market": [],
    "confidentiality_clause": "",
    "internet_rate_restriction": ""
  }},

  "room_types": [
    {{
      "room_name": "",
      "room_code": "",
      "room_category": "",
      "description": "",
      "size_sqm": "",
      "number_of_rooms": "",
      "bed_types": [],
      "view": "",
      "window_type": "",
      "max_occupancy": "",
      "standard_occupancy": "",
      "room_features": [],

      "seasonal_rates": [
        {{
          "season_name": "",
          "date_range": "",
          "occupancy_type": "",
          "market_type": "FIT",
          "rate_plan": "",
          "currency": "",
          "price": "",
          "meal_included": [],
          "tax_included": "",
          "service_charge_included": "",
          "rate_notes": []
        }}
      ]
    }}
  ],

  "extra_bed_policy": {{
    "available": "",
    "max_extra_beds_per_room": "",

    "extra_bed_rates": [
      {{
        "season": "",
        "currency": "",
        "price": "",
        "conditions": []
      }}
    ]
  }},

  "child_policy": {{

    "infant_policy": [
      {{
        "age_range": "",
        "currency": "",
        "price": "",
        "conditions": []
      }}
    ],

    "child_free_policy": [
      {{
        "age_range": "",
        "currency": "",
        "price": "",
        "conditions": []
      }}
    ],

    "child_breakfast_policy": [
      {{
        "age_range": "",
        "currency": "",
        "price": "",
        "conditions": []
      }}
    ],

    "child_extra_bed_policy": [
      {{
        "age_range": "",
        "currency": "",
        "price": "",
        "conditions": []
      }}
    ],

    "adult_definition": "",
    "child_definition": "",

    "baby_cot_policy": [
      {{
        "currency": "",
        "price": "",
        "conditions": []
      }}
    ]
  }},

  "meal_information": {{

    "restaurant_information": [],

    "meal_rates": [
      {{
        "meal_type": "",
        "currency": "",
        "price": "",
        "conditions": []
      }}
    ]
  }},

  "included_benefits": [
    {{
      "benefit_type": "",
      "details": "",
      "applicable_room_types": [],
      "conditions": []
    }}
  ],

  "facilities": [
    {{
      "facility_name": "",
      "availability": "",
      "notes": ""
    }}
  ],

  "transportation_services": [
    {{
      "service_type": "",
      "vehicle_type": "",
      "route": "",
      "currency": "",
      "price": "",
      "conditions": []
    }}
  ],

  "checkin_checkout_policy": {{

    "checkin_time": "",
    "checkout_time": "",

    "early_checkin_policy": [
      {{
        "time_range": "",
        "currency": "",
        "price": "",
        "conditions": []
      }}
    ],

    "late_checkout_policy": [
      {{
        "time_range": "",
        "currency": "",
        "price": "",
        "conditions": []
      }}
    ]
  }},

  "cancellation_policy": [
    {{
      "period": "",
      "currency": "",
      "price": "",
      "percentage": "",
      "conditions": []
    }}
  ],

  "payment_policy": {{

    "payment_terms": [
      {{
        "details": ""
      }}
    ],

    "deposit_policy": [
      {{
        "details": ""
      }}
    ],

    "refund_policy": [
      {{
        "details": ""
      }}
    ]
  }},

  "peak_period_surcharge": [
    {{
      "occasion": "",
      "date_range": "",
      "currency": "",
      "price": "",
      "percentage": "",
      "conditions": []
    }}
  ],

  "restrictions": [
    {{
      "type": "",
      "details": ""
    }}
  ],

  "important_notes": [],

  "raw_season_definitions": [
    {{
      "season_name": "",
      "date_ranges": []
    }}
  ],

  "special_hotel_feature": {{
    "feature_name": "",
    "feature_category": "",
    "details": ""
  }},

  "metadata": {{
    "document_language": [],
    "document_type": "",
    "extraction_confidence": "",
    "contains_fit_rates": "",
    "contains_child_policy": "",
    "contains_meal_policy": "",
    "contains_transport_service": "",
    "contains_peak_surcharge": "",
    "contains_special_feature": ""
  }}
}}

==================================================
DOCUMENT
==================================================

{pdf_text}

"""

# =====================================================
# OLLAMA CALL
# =====================================================

response = chat(
    model='qwen2.5:7b',
    format='json',
    messages=[
        {
            'role': 'user',
            'content': PROMPT
        }
    ]
)

# =====================================================
# RESULT
# =====================================================

result = response['message']['content']

print("=" * 50)
print("MODEL RESPONSE")
print("=" * 50)

print(result)

# =====================================================
# SAVE RAW RESPONSE
# =====================================================

with open("raw_response.txt", "w", encoding="utf-8") as f:
    f.write(result)

print("Saved -> raw_response.txt")

# =====================================================
# JSON VALIDATION
# =====================================================

try:

    parsed_json = json.loads(result)

    with open("output.json", "w", encoding="utf-8") as f:
        json.dump(parsed_json, f, ensure_ascii=False, indent=2)

    print("SUCCESS -> output.json")

except Exception as e:

    print("=" * 50)
    print("JSON PARSE ERROR")
    print("=" * 50)

    print(e)

    print("\nModel returned invalid JSON.")
    print("Check raw_response.txt")

In [ ]:
import json
import pdfplumber
from ollama import chat

# =====================================================
# FILE PATH
# =====================================================

FILE_PATH = r"D:\365\data\BẢNG-GIÁ-KHUYẾN-MÃI-GIAI-ĐOẠN-10.02-ĐẾN-31.03.2023 (1).pdf"

# =====================================================
# READ PDF
# =====================================================

def read_pdf(pdf_path):

    text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    return text

# =====================================================
# LOAD PDF TEXT
# =====================================================

pdf_text = read_pdf(FILE_PATH)

print("=" * 50)
print("PDF TEXT PREVIEW")
print("=" * 50)

print(pdf_text[:3000])

# =====================================================
# PROMPT
# =====================================================

PROMPT = f"""
You are an expert hospitality FIT contract extraction engine.

Your task is to extract ONLY FIT (Free Independent Traveler)
hotel contract information from the uploaded document.

Return STRICT VALID JSON ONLY.

==================================================
CORE EXTRACTION RULES
==================================================

1. ONLY extract information explicitly written in the document.

2. DO NOT hallucinate, infer, estimate, normalize, or invent values.

3. IGNORE all NON-FIT information:
- GIT
- GROUP
- SERIES
- CORPORATE
- WHOLESALE
- MICE
- EVENT
- ALLOTMENT
- CREW
- CONTRACTOR
- OTA CAMPAIGN

4. If a table contains both FIT and NON-FIT data:
extract ONLY FIT rows and columns.

5. USE ONLY the EXACT schema keys defined below.

6. DO NOT:
- create additional keys
- rename keys
- restructure arrays
- restructure objects
- generate nested attributes not defined in schema

7. NEVER generate undefined keys such as:
- age_bracket
- breakfast_fee
- room_rate_fit
- policy_item
- surcharge_data
- hotel_package
- package_detail
- extra_information

8. Preserve EXACTLY:
- room names
- currencies
- VND formatting
- percentages
- date ranges
- rate formatting
- punctuation
- multilingual wording

9. If information is missing:
- use empty string ""
- use empty array []
- NEVER guess

10. NEVER explain.

11. NEVER summarize.

12. NEVER return markdown.

13. NEVER wrap JSON inside code blocks.

14. Schema consistency is CRITICAL across all hotels.

15. Return STRICT VALID JSON ONLY.

==================================================
SEMANTIC RULES
==================================================

A. facilities
= permanent hotel facilities/amenities.

Examples:
- swimming pool
- gym
- spa
- sauna
- wifi
- parking
- beach access
- kids club

B. included_benefits
= benefits included with booking/rate/package.

Examples:
- breakfast included
- airport transfer
- welcome drink
- complimentary dinner
- spa voucher
- late checkout

DO NOT duplicate facilities into included_benefits.

C. meal_information
ONLY use for:
- restaurant information
- gala dinner
- compulsory dinner
- buffet pricing
- special meal pricing

DO NOT duplicate breakfast inclusion here
if breakfast already exists inside room seasonal_rates.

Breakfast inclusion inside room seasonal_rates
is the PRIMARY SOURCE OF TRUTH.

==================================================
OUTPUT STRUCTURE
==================================================

{{
  "hotel_information": {{
    "hotel_name": "",
    "brand_group": "",
    "address": "",
    "district": "",
    "city": "",
    "country": "",
    "website": "",
    "email": [],
    "phone_numbers": [],
    "tax_code": "",
    "representatives": [
      {{
        "name": "",
        "position": ""
      }}
    ]
  }},

  "contract_information": {{
    "contract_name": "",
    "contract_number": "",
    "validity": {{
      "start_date": "",
      "end_date": ""
    }},
    "applicable_market": [],
    "confidentiality_clause": "",
    "internet_rate_restriction": ""
  }},

  "room_types": [
    {{
      "room_name": "",
      "room_code": "",
      "room_category": "",
      "description": "",
      "size_sqm": "",
      "number_of_rooms": "",
      "bed_types": [],
      "view": "",
      "window_type": "",
      "max_occupancy": "",
      "standard_occupancy": "",
      "room_features": [],

      "seasonal_rates": [
        {{
          "season_name": "",
          "date_range": "",
          "occupancy_type": "",
          "market_type": "FIT",
          "rate_plan": "",
          "currency": "",
          "price": "",
          "meal_included": [],
          "tax_included": "",
          "service_charge_included": "",
          "rate_notes": []
        }}
      ]
    }}
  ],

  "extra_bed_policy": {{
    "available": "",
    "max_extra_beds_per_room": "",

    "extra_bed_rates": [
      {{
        "season": "",
        "currency": "",
        "price": "",
        "conditions": []
      }}
    ]
  }},

  "child_policy": {{

    "infant_policy": [
      {{
        "age_range": "",
        "currency": "",
        "price": "",
        "conditions": []
      }}
    ],

    "child_free_policy": [
      {{
        "age_range": "",
        "currency": "",
        "price": "",
        "conditions": []
      }}
    ],

    "child_breakfast_policy": [
      {{
        "age_range": "",
        "currency": "",
        "price": "",
        "conditions": []
      }}
    ],

    "child_extra_bed_policy": [
      {{
        "age_range": "",
        "currency": "",
        "price": "",
        "conditions": []
      }}
    ],

    "adult_definition": "",
    "child_definition": "",

    "baby_cot_policy": [
      {{
        "currency": "",
        "price": "",
        "conditions": []
      }}
    ]
  }},

  "meal_information": {{

    "restaurant_information": [],

    "meal_rates": [
      {{
        "meal_type": "",
        "currency": "",
        "price": "",
        "conditions": []
      }}
    ]
  }},

  "included_benefits": [
    {{
      "benefit_type": "",
      "details": "",
      "applicable_room_types": [],
      "conditions": []
    }}
  ],

  "facilities": [
    {{
      "facility_name": "",
      "availability": "",
      "notes": ""
    }}
  ],

  "transportation_services": [
    {{
      "service_type": "",
      "vehicle_type": "",
      "route": "",
      "currency": "",
      "price": "",
      "conditions": []
    }}
  ],

  "checkin_checkout_policy": {{

    "checkin_time": "",
    "checkout_time": "",

    "early_checkin_policy": [
      {{
        "time_range": "",
        "currency": "",
        "price": "",
        "conditions": []
      }}
    ],

    "late_checkout_policy": [
      {{
        "time_range": "",
        "currency": "",
        "price": "",
        "conditions": []
      }}
    ]
  }},

  "cancellation_policy": [
    {{
      "period": "",
      "currency": "",
      "price": "",
      "percentage": "",
      "conditions": []
    }}
  ],

  "payment_policy": {{

    "payment_terms": [
      {{
        "details": ""
      }}
    ],

    "deposit_policy": [
      {{
        "details": ""
      }}
    ],

    "refund_policy": [
      {{
        "details": ""
      }}
    ]
  }},

  "peak_period_surcharge": [
    {{
      "occasion": "",
      "date_range": "",
      "currency": "",
      "price": "",
      "percentage": "",
      "conditions": []
    }}
  ],

  "restrictions": [
    {{
      "type": "",
      "details": ""
    }}
  ],

  "important_notes": [],

  "raw_season_definitions": [
    {{
      "season_name": "",
      "date_ranges": []
    }}
  ],

  "special_hotel_feature": {{
    "feature_name": "",
    "feature_category": "",
    "details": ""
  }},

  "metadata": {{
    "document_language": [],
    "document_type": "",
    "extraction_confidence": "",
    "contains_fit_rates": "",
    "contains_child_policy": "",
    "contains_meal_policy": "",
    "contains_transport_service": "",
    "contains_peak_surcharge": "",
    "contains_special_feature": ""
  }}
}}

==================================================
DOCUMENT
==================================================

{pdf_text}

"""

# =====================================================
# OLLAMA CALL
# =====================================================

response = chat(
    model='tinyllama',
    format='json',
    messages=[
        {
            'role': 'user',
            'content': PROMPT
        }
    ]
)

# =====================================================
# RESULT
# =====================================================

result = response['message']['content']

print("=" * 50)
print("MODEL RESPONSE")
print("=" * 50)

print(result)

# =====================================================
# SAVE RAW RESPONSE
# =====================================================

with open("raw_response.txt", "w", encoding="utf-8") as f:
    f.write(result)

print("Saved -> raw_response.txt")

# =====================================================
# JSON VALIDATION
# =====================================================

try:

    parsed_json = json.loads(result)

    with open("output.json", "w", encoding="utf-8") as f:
        json.dump(parsed_json, f, ensure_ascii=False, indent=2)

    print("SUCCESS -> output.json")

except Exception as e:

    print("=" * 50)
    print("JSON PARSE ERROR")
    print("=" * 50)

    print(e)

    print("\nModel returned invalid JSON.")
    print("Check raw_response.txt")